# Physics Review: 1-D Transient Chimney Simulation

**Package**: `flued_module` (PyFluesim)  
**Branch**: `physics-fixes-transient-debug`  
**Date**: February 2026  

---

## How to read this notebook

**Intended audience:** Physics reviewer and/or developer who needs to understand *exactly* what equations are solved, in what order, and what assumptions are made.

**Scope:** This document describes the **currently implemented physics** in the codebase. It is not aspirational. Every equation, correlation, and algorithmic choice described here has a corresponding line of code, referenced by `module:line_number` where helpful.

**Conventions:**
- SI units throughout unless stated otherwise.
- Temperatures in Kelvin (K) unless marked with `[degC]`.
- Subscript `seg` = segment index (0 at bottom inlet, `N_seg - 1` at top outlet).
- Superscript `n` = timestep index; `k` = Picard iteration index within a timestep.
- Positive mass flow = upward (normal draft); negative = downward (backdraft).

---

# 1. Executive Summary

## State variables (per segment, per timestep)
- $T_\text{gas}$ -- gas temperature [K]
- $T_\text{wall}$ -- wall temperature (single-wall) or $T_\text{liner}, T_\text{middle}, T_\text{outer}$ (triple-wall) [K]
- $\rho$ -- gas density [kg/m$^3$]
- $v$ -- gas velocity [m/s]
- $P$ -- static pressure [Pa]
- $\dot{m}$ -- mass flow rate [kg/s] (scalar, same for all segments at a given timestep)

## Governing conservation laws
- **Mass**: 1-D continuity with optional distributed leakage (VENT-II style orifice model).
- **Momentum**: Quasi-steady algebraic balance with first-order lag, **or** unsteady momentum ODE (semi-implicit Euler). Buoyancy drives flow; friction, fittings, inlet restriction, and optional acceleration losses resist it.
- **Energy (gas)**: Advection (upwind scheme) + convective wall heat loss + optional condensation latent heat. Two gas energy sub-models: (a) relaxation-based, (b) plug-flow NTU/effectiveness.
- **Energy (wall)**: Transient lumped-capacitance for single wall (semi-implicit). 3-node thermal network for triple wall (implicit Euler or adaptive Gauss-Seidel sub-stepping), including inter-layer radiation.

## Coupling approach
- Picard (fixed-point) iteration couples $T \to \rho \to \Delta P_\text{buoyancy} \to \dot{m} \to h \to T$.
- Default: up to 5 iterations/timestep, tolerance $10^{-3}$ on relative residuals (temperature, flow, pressure).
- Adaptive under-relaxation: $\alpha \in [0.2, 0.5]$, adjusted based on residual trend.

## Major correlations
- Ideal gas density: $\rho = P / (R_\text{air} T)$, $R_\text{air} = 287$ J/(kg K).
- Gas properties: Sutherland viscosity, empirical $k(T)$ and $c_p(T)$ polynomials.
- Forced convection: Gnielinski/Dittus-Boelter for turbulent; Hausen for developing laminar.
- Natural convection: Churchill-Chu type correlations (internal and external).
- Combined convection: $h = (h_\text{forced}^3 + h_\text{natural}^3)^{1/3}$.
- Friction: Laminar $f = 64/\text{Re}$; turbulent Blasius (smooth) or Swamee-Jain (rough); optional iterative Colebrook-White.
- Condensation: Dew-point tracking, film-modified $h$, and latent heat source.
- Radiation: Linearized inter-layer radiation for triple wall; $h_\text{rad}$ for outer-to-ambient.

## What happens each timestep (one paragraph)

At each timestep, boundary conditions are first updated (door opening, room plenum pressure, inlet temperature). The coupled solver then enters a Picard loop: it updates gas densities from the ideal gas law, computes per-segment buoyancy pressure from density differences with ambient, sums driving pressure (buoyancy $-$ wind cap $-$ room depressurization), accumulates resistances (pipe friction + fittings + inlet restriction + optional catalytic bypass + optional acceleration loss + optional leakage), solves the momentum equation to obtain a new mass flow rate and pressure profile, computes segment velocities from continuity, then marches through all segments to solve the gas energy equation (upwind advection with wall heat loss) while simultaneously updating wall temperatures via the comprehensive heat transfer module (which handles inner/outer convection, radiation, condensation, and single- or triple-wall transient conduction). Residuals are computed, relaxation is applied to all state variables, and if all residuals are below tolerance the iteration exits; otherwise the loop repeats. The converged state is then committed to the output arrays.

---

# 2. Full Physics Description

## 2A. Geometry and Discretization

### Part vs. segment mapping

The chimney is defined as an ordered list of **parts** (pipe sections), each with its own diameter, wall type (single/triple), material properties, height, orientation (vertical/horizontal), elevation, and ambient conditions. Each part is subdivided into `num_segments` equal-length **segments**. Segments are the fundamental computational cells.

Global segment index `seg` maps to a `(part_idx, local_seg)` pair via `chimney.get_segment_info(seg)` (`chimney.py:1875`).

### Segment geometry

| Quantity | Formula | Stored in |
|---|---|---|
| Inner diameter $D_i$ | Converted from ft to m in `__post_init__` | `part.inner_diameter_m` |
| Inner cross-section area $A$ | $\pi D_i^2 / 4$ | `part.inner_area_m2` |
| Segment length $\Delta z$ | `part.height_m / part.num_segments` | `chimney.segment_lengths[seg]` |
| Inner surface area per segment | $\pi D_i \Delta z$ | Computed inline |
| Wall volume (single) | $\pi \Delta z (D_o^2 - D_i^2)/4$ | Computed in `transient_wall_temperature` |

For triple-wall parts, intermediate radii are computed from liner/insulation/outer thicknesses plus optional air gaps:

$$r_\text{gas} = D_i/2, \quad r_1 = r_\text{gas} + t_\text{liner}, \quad r_2 = r_1 + t_\text{middle}, \quad r_3 = r_2 + t_\text{outer}$$

### State variables stored per segment per timestep

All state arrays have shape `(time_steps + 1, total_segments)` and are pre-allocated in `_initialize_arrays()` (`chimney.py:1760`):

| Array | Symbol | Units | Description |
|---|---|---|---|
| `T_gas` | $T_g$ | K | Gas temperature |
| `T_wall` | $T_w$ | K | Wall temperature (for single-wall; equals $T_\text{liner}$ for triple) |
| `T_liner` | $T_1$ | K | Triple-wall inner liner temperature |
| `T_middle` | $T_2$ | K | Triple-wall insulation temperature |
| `T_outer` | $T_3$ | K | Triple-wall outer shell temperature |
| `density` | $\rho$ | kg/m$^3$ | Gas density |
| `velocity` | $v$ | m/s | Gas velocity |
| `pressure` | $P$ | Pa | Static pressure |
| `h_conv_inner` | $h_i$ | W/(m$^2$ K) | Inner convection coefficient |
| `h_conv_outer` | $h_o$ | W/(m$^2$ K) | Outer convection coefficient |

Scalars per timestep: `mass_flow_rate` [kg/s], `door_opening_history` [%].

### Initial conditions (`chimney.py:1804`)

- Gas: segment 0 gets inlet temperature (from function, CSV, or $T_\text{amb} + 10$ K); all other segments start at their local $T_\text{ambient}$.
- Walls: all layers initialized to $T_\text{ambient}$.
- Density: from ideal gas law at initial $T_g$.
- Velocity: 0.1 m/s everywhere.
- Mass flow: 0.01 kg/s.
- Pressure: atmospheric everywhere.

## 2B. Gas Properties and Thermodynamics

All gas property correlations are in `physics.py` (class `PhysicalProperties`). The gas is treated as **dry air** throughout (a good approximation for flue gas in most applications, per the module docstring).

### Density (ideal gas law)

$$\rho = \frac{P}{R_\text{air} \, T}, \qquad R_\text{air} = 287 \; \text{J/(kg K)}$$

Called via `PhysicalProperties.air_density(T, P)` (`physics.py:23`). Default pressure is `P_STD = 98457` Pa (lab elevation -- intentional, not sea level).

### Dynamic viscosity (Sutherland's formula)

$$\mu(T) = \mu_\text{ref} \left(\frac{T}{T_\text{ref}}\right)^{3/2} \frac{T_\text{ref} + S}{T + S}$$

with $\mu_\text{ref} = 1.716 \times 10^{-5}$ Pa s, $T_\text{ref} = 273.15$ K, $S = 110.4$ K.

Called via `air_viscosity(T)` (`physics.py:42`).

### Thermal conductivity

$$k(T) = \frac{2.624 \times 10^{-3} \, T^{3/2}}{T + 245.4 \times 10^{-12/T}}$$

Empirical correlation valid ~100-1000 K. Called via `air_thermal_conductivity(T)` (`physics.py:65`).

### Specific heat at constant pressure

$$c_p(T) = 1030.5 - 0.19975\, T + 4.2775 \times 10^{-4}\, T^2 - 2.4223 \times 10^{-7}\, T^3$$

Third-order polynomial fit, valid ~200-800 K. Called via `air_specific_heat(T)` (`physics.py:84`).

### Ceramic fiber insulation conductivity

For triple-wall insulation, $k$ is temperature-dependent:

$$k(T) = k_\text{ref} + \alpha\,(T - T_\text{ref}) + \beta\,(T - T_\text{ref})^2$$

with $k_\text{ref} = 0.04$ W/(m K) at 20 degC, $\alpha = 1.8 \times 10^{-4}$, $\beta = 1.5 \times 10^{-7}$, capped at 0.30 W/(m K). Called via `ceramic_fiber_thermal_conductivity(T)` (`physics.py:104`).

### Where properties are evaluated

In the coupled solver, gas properties are evaluated at the **current iteration's temperature** for each segment. The density update (`_update_densities`, `coupled_solver.py:309`) uses each segment's $T_g$ and the part's atmospheric pressure. Viscosity is cached per segment during the momentum solve to avoid redundant evaluations.

## 2C. Buoyancy / Draft Driving Pressure

### Per-segment buoyancy

Buoyancy pressure is computed per segment in `_calculate_buoyancy()` (`coupled_solver.py:320`):

$$\Delta P_\text{buoy,seg} = g \, \Delta z_\text{seg} \, (\rho_\text{ambient} - \rho_\text{gas,seg})$$

where $g = 9.81$ m/s$^2$, $\Delta z$ is the segment length, and $\rho_\text{ambient}$ is computed at the part's ambient temperature and pressure. Horizontal segments contribute zero buoyancy (their $\Delta z$ is the segment length, but orientation is accounted for by using the actual vertical height component).

### Total driving pressure

The net driving pressure entering the momentum balance (`coupled_solver.py:578`) is:

$$\Delta P_\text{drive} = \sum_\text{seg} \Delta P_\text{buoy,seg} \;-\; \Delta P_\text{wind} \;-\; \Delta P_\text{room}$$

**Additional terms:**

- **Wind cap $\Delta P_\text{wind}$**: Computed from `wind_cap_deltaP()` (`wind_cap_effects.py:112`). Physics: $\Delta P = C_p \cdot \tfrac{1}{2}\rho V_\text{wind}^2 \cdot G$, where $C_p$ depends on cap geometry (`open`, `rain_cap`, `venturi`, etc.), roof exposure, and incidence angle. Negative $C_p$ (suction) *aids* draft; the sign convention is that `deltaP_wind` is *subtracted* from driving pressure, so suction adds to it.

- **Room depressurization $\Delta P_\text{room}$**: When the room plenum model is active (`room_plenum.py`), the chimney draws air from the room, lowering its pressure below ambient. The room is modeled as a pneumatic capacitance:
$$\frac{dP_\text{room}}{dt} = \frac{(\dot{m}_\text{leak,in} - \dot{m}_\text{chimney}) \, R_\text{air} \, T_\text{room}}{V_\text{room}}$$
where $\dot{m}_\text{leak,in} = C_d A_\text{room,leak} \sqrt{2\rho|P_\text{amb} - P_\text{room}|}$. This is integrated with adaptive subcycling (forward Euler). The resulting room depressurization reduces chimney draft.

## 2D. Momentum / Flow Model

### Two flow models

Controlled by `MomentumConfig.flow_model` (default `"quasi_steady"`):

#### Quasi-steady (default)

Algebraic steady-state mass flow with a first-order lag toward it (`coupled_solver.py:781`):

$$\dot{m}_\text{ss} = \text{sign}(\Delta P_\text{drive}) \sqrt{\frac{|\Delta P_\text{drive}|}{R_\text{total}}}$$

$$\tau = \frac{\bar{\rho} \, L_\text{system}}{R_\text{total} \, (|\dot{m}_\text{ss}| + 10^{-6})}$$

$$\alpha = \min\left(\frac{\Delta t}{\tau + \Delta t}, \; 0.5\right)$$

$$\dot{m}^{n+1} = \dot{m}^n + \alpha\,(\dot{m}_\text{ss} - \dot{m}^n)$$

The lag $\tau$ prevents instantaneous jumps; $\alpha$ is capped at 0.5 to prevent overshooting.

#### Unsteady momentum ODE

A proper inertial ODE (`coupled_solver.py:754`):

$$\frac{L_\text{eff}}{A} \frac{d\dot{m}}{dt} = \Delta P_\text{drive} - R_\text{total} \, \dot{m}\,|\dot{m}|$$

Discretized with semi-implicit Euler (unconditionally stable):

$$\dot{m}^{n+1} = \frac{\dot{m}^n + C \cdot \Delta P_\text{drive}}{1 + C \cdot R_\text{total} \cdot |\dot{m}^n|}$$

where $C = \Delta t \cdot A / L_\text{eff}$. **Important**: the ODE always integrates from the **previous timestep's** $\dot{m}$ (not the Picard iterate), so the starting point is fixed within a timestep.

### Resistance model

Total quadratic resistance $R_\text{total}$ is the sum of five components:

#### 1. Pipe friction ($R_\text{friction}$)

Per segment (`coupled_solver.py:615`):

$$R_\text{seg} = \frac{f \cdot L/D + K_\text{minor}}{2 \rho A^2}$$

**Friction factor regime logic** (`mass_flow.py:18` and inline in `coupled_solver.py`):

| Regime | Criterion | Correlation |
|---|---|---|
| Laminar | $\text{Re} < 2300$ | $f = 64/\text{Re}$ |
| Turbulent, smooth ($\varepsilon/D < 10^{-6}$) | $\text{Re} \ge 2300$ | Blasius: $f = 0.316/\text{Re}^{0.25}$ |
| Turbulent, rough | $\text{Re} \ge 2300$ | Swamee-Jain: $f = 0.25/[\log_{10}(\varepsilon/(3.7D) + 5.74/\text{Re}^{0.9})]^2$ |
| (Optional) | $\text{Re} \ge 2300$ | Iterative Colebrook-White (if `friction_method="colebrook"`) |

Reynolds number is floored at 100 (or 10 in the pressure profile update) to avoid division by zero.

#### 2. Minor losses (fittings/elbows)

Elbows are defined as `ElbowSpec` objects with angle, R/D, type, smoothness, and vane factor. The K-factor is `base_K(spec) * spec.vane_factor`. Pressure drop:

$$\Delta P_\text{minor} = K_\text{elbow} \cdot \tfrac{1}{2} \rho v^2$$

#### 3. Inlet restriction resistance

The door/damper at the chimney inlet is modeled as an orifice (`coupled_solver.py:341`):

$$R_\text{door} = \frac{K_\text{equiv}}{2 \rho_\text{inlet} A_\text{eff}^2}, \qquad K_\text{equiv} = 1/C_d^2$$

where $A_\text{eff}$ is the effective flow area combining geometric opening and leakage:
- **Linear mode** (default): $A_\text{eff} = A_\text{raw} + A_\text{leak}$
- **Quadrature mode**: $A_\text{eff} = \sqrt{A_\text{raw}^2 + A_\text{leak}^2}$

If `K_stove > 0`, an additional series resistance for the stove body is added: $R_\text{stove} = K_\text{stove}/(2\rho A_\text{pipe}^2)$.

#### 4. Distributed leakage (VENT-II style, optional)

If `MomentumConfig.use_distributed_leakage` is `True` (`mass_flow.py:101`):

$$\dot{m}_\text{leak} = -\text{sign}(P_\text{gauge}) \cdot C_d A_\text{leak} \sqrt{2 \rho_\text{ref} |P_\text{gauge}|}$$

where $P_\text{gauge} = P_\text{seg} - P_a$. Positive gauge pressure $\to$ outflow $\to$ negative leak (reduces downstream mass). Iterated `N_leak_iter` times with under-relaxation `leak_relaxation` (default 0.5). The total leakage mass flow modifies $\dot{m}$ additively (not as a resistance).

#### 5. Acceleration loss (optional)

If `MomentumConfig.use_accel_loss` is `True` (`mass_flow.py:148`):

$$\Delta P_\text{accel} = C_\text{accel} \sum_{\text{seg}} \left(\tfrac{1}{2}\rho_\text{down} v_\text{down}^2 - \tfrac{1}{2}\rho_\text{up} v_\text{up}^2\right)$$

This accounts for the velocity head change as gas heats up (density decreases, velocity increases). Subtracted from $\Delta P_\text{drive}$.

### Mass flow clipping

After the momentum solve, $|\dot{m}|$ is clipped to $v_\text{max} \cdot \bar{\rho} \cdot A_\text{typical}$ with $v_\text{max} = 50$ m/s (`coupled_solver.py:806`).

## 2E. Energy Model for Gas

The gas energy is solved segment-by-segment in `_solve_energy_improved()` (`coupled_solver.py:912`). There are **two gas energy sub-models**, controlled by `MomentumConfig.gas_energy_model`.

### Upwind advection scheme

The upstream temperature $T_\text{up}$ depends on flow direction (`coupled_solver.py:1006`):

| Flow direction | $T_\text{up}$ for segment `seg` |
|---|---|
| Normal draft ($\dot{m} \ge 0$) | $T_\text{gas}[\text{seg}-1]$ (segment below); segment 0 uses $T_\text{inlet}$ |
| Backdraft ($\dot{m} < 0$) | $T_\text{gas}[\text{seg}+1]$ (segment above); top segment uses $T_\text{ambient}$ |

### Model A: "current" (default) -- Relaxation-based

For flowing gas ($|\dot{m}| > 10^{-6}$, non-inlet boundary) (`coupled_solver.py:1113`):

$$q_\text{loss} = h_i A_s (T_g - T_w) + q_\text{latent}$$

$$\tau_\text{res} = m_\text{gas} / |\dot{m}|$$

$$T_\text{steady} = T_\text{up} - \frac{q_\text{loss} \cdot \tau_\text{res}}{m_\text{gas} \cdot c_p}$$

where $m_\text{gas} = \rho^{n-1} \cdot A \cdot \Delta z$ is the gas mass in the segment. $T_\text{steady}$ is then **bounded** to $[\min(T_\text{up}, T_w) - 1, \; \max(T_\text{up}, T_w) + 1]$ K to prevent divergence during flow reversal.

Finally, relaxation toward $T_\text{steady}$:

$$\tau_\text{thermal} = \frac{m_\text{gas} \, c_p}{|\dot{m}| c_p + h_i A_s}$$

$$\alpha_T = \frac{\Delta t}{\tau_\text{thermal} + \Delta t}$$

$$T_g^{\text{new}} = T_g^{\text{old}} + \alpha_T (T_\text{steady} - T_g^{\text{old}})$$

### Model B: "plugflow_ntu" -- NTU/Effectiveness

Treats each segment as a counter-flow heat exchanger (`coupled_solver.py:1083`):

$$\text{UA} = h_i \cdot A_s$$

$$\text{NTU} = \text{UA} / (|\dot{m}| \, c_p)$$

$$\varepsilon = 1 - e^{-\text{NTU}}$$

$$T_\text{out} = T_\text{up} - \varepsilon \cdot (T_\text{up} - T_w) - q_\text{latent}/(|\dot{m}| \, c_p)$$

### No/low-flow regime ($|\dot{m}| \le 10^{-6}$)

Gas equilibrates toward wall temperature (`coupled_solver.py:1151`):

$$\tau = \frac{m_\text{gas} \, c_p}{h_i A_s}, \qquad T_g^{\text{new}} = T_g^{\text{old}} + \frac{T_w - T_g^{\text{old}}}{\tau} \Delta t$$

### Inlet boundary

Inlet segment temperature is **fixed** at the boundary condition value (fire temperature for normal draft; ambient for backdraft).

### Physical bounds

For non-inlet segments (`coupled_solver.py:1159`):
- Floor: $T_\text{ambient}$ (gas cannot cool below ambient in a passive system)
- Ceiling: 1073.15 K (800 degC)

These are **numerical safety clips**, not physics constraints. The floor prevents unphysical sub-ambient gas temperatures that could arise from numerical overshoot. The ceiling prevents runaway in poorly converged iterations.

## 2F. Heat Transfer to Walls

The comprehensive heat transfer module (`comprehensive_heat_transfer.py`) is called per segment per Picard iteration via `calculate_segment_heat_transfer(segment_data, dt)` (line 711).

### Inner convection coefficient $h_i$

**Forced convection** (from gas flow):
- $\text{Re} = \rho |v| D / \mu$, $\text{Pr} = \mu c_p / k$
- Nusselt number from `nusselt_forced_convection_internal(Re, Pr, L/D)` -- uses Gnielinski or Dittus-Boelter type correlation for turbulent, Hausen for developing laminar.
- $h_\text{forced} = \text{Nu}_\text{forced} \cdot k / D$
- Fittings enhancement: $h_\text{forced} \times 1.2$ if segment has fittings.

**Natural convection** (from temperature differences):
- Grashof number: $\text{Gr} = g \beta \Delta T D^3 / \nu^2$, $\beta = 1/T_\text{mean}$ for ideal gas.
- Rayleigh number: $\text{Ra} = \text{Gr} \cdot \text{Pr}$
- $\text{Nu}_\text{natural}$ from `nusselt_natural_convection_internal(Ra, orientation, L/D)`.
- $h_\text{natural} = \text{Nu}_\text{natural} \cdot k / D$

**Combined convection:**

$$h_i = (h_\text{forced}^3 + h_\text{natural}^3)^{1/3}$$

via `combined_convection_coefficient()` -- this is the asymptotic correlation for combined forced + natural convection.

**Engineering minimums** (`comprehensive_heat_transfer.py:813`):
- Vertical chimney with hot gas ($T_g > 373$ K): $h_i \ge 10$ W/(m$^2$ K)
- Other orientations: $h_i \ge 5$ W/(m$^2$ K)

These minimums account for turbulence, roughness, and entrance effects not captured by smooth-pipe correlations.

### Outer convection coefficient $h_o$

- Natural convection on external surface: Grashof/Rayleigh at outer diameter with $\Delta T = T_\text{wall,outer} - T_\text{ambient}$.
- Radiation to ambient: $h_\text{rad} = \epsilon \sigma (T_w^2 + T_\text{amb}^2)(T_w + T_\text{amb})$
- Total: $h_o = h_\text{conv,outer} + h_\text{rad}$
- Floor: $h_o \ge 1.0$ W/(m$^2$ K)

### Corrugation enhancement

If spiral corrugation is configured with `enable_spiral_HT_enhancement`, the inner convection coefficient is multiplied by a swirl-induced enhancement factor $M_h$ (typically 1.0-1.25, capped at 25%). This is pre-computed during geometry setup (`corrugation_effects.py`).

### Condensation physics

If the wall temperature $T_w$ drops below the gas dew point $T_\text{dew}$ (`condensation_physics.py`):

1. **Dew point** is calculated from saturation pressure (Buck equation for 0-100 degC) and humidity ratio.
2. **Condensation rate** is computed from the mass transfer driving force (difference between gas humidity and saturation humidity at wall temperature).
3. **Film thickness** grows from condensation and shrinks from evaporation.
4. **Latent heat rate** $q_\text{latent} = \dot{m}_\text{condensation} \cdot h_{fg}$ is added to the wall heat source.
5. **Film modification** of $h_i$: condensate film provides an additional thermal resistance.

The condensation model iterates up to 5 times within each HT kernel call to converge the coupled wall-temperature / condensation state.

## 2G. Wall Thermal Model

### Single wall: semi-implicit lumped capacitance

Computed in `transient_wall_temperature()` (`comprehensive_heat_transfer.py:524`).

The wall is treated as a lumped mass with cylindrical geometry:

$$C_w = \rho_w c_{p,w} V_w, \qquad V_w = \frac{\pi \Delta z}{4}(D_o^2 - D_i^2)$$

Thermal conductances:
- Inner: $\text{UA}_i = h_i \cdot \pi D_i \Delta z$
- Outer: $\text{UA}_o = h_o \cdot \pi D_o \Delta z$

**Semi-implicit time integration** (backward Euler on conductance terms):

$$\left(\frac{C_w}{\Delta t} + \text{UA}_i + \text{UA}_o\right) T_w^{n+1} = \frac{C_w}{\Delta t} T_w^n + \text{UA}_i \, T_g + \text{UA}_o \, T_\text{amb}$$

$$\Rightarrow \quad T_w^{n+1} = \frac{C_w/\Delta t \cdot T_w^n + \text{UA}_i \, T_g + \text{UA}_o \, T_\text{amb}}{C_w/\Delta t + \text{UA}_i + \text{UA}_o}$$

This is **unconditionally stable** (no CFL restriction on $\Delta t$). Additional damping (0.7 factor blending with $T_w^n$) is applied only for very small timesteps ($\Delta t < 0.01$ s).

**Bounds**: $T_w \in [T_\text{amb}, T_g]$ for hot gas above ambient; $T_w \ge T_\text{amb}$ otherwise.

### Triple wall: 3-node thermal network

Three computational nodes: liner ($T_1$), insulation ($T_2$), outer shell ($T_3$).

**Two solver paths** (`comprehensive_heat_transfer.py:658`):

1. **Adaptive sub-stepping** (`adaptive_triple_wall.py`): Used when liner is very thin ($< 1$ mm) or $\Delta t > 0.05$ s (stiff system). Estimates required sub-step from layer time constants:

$$\tau_i = C_i / G_i^\text{max}, \qquad \Delta t_\text{req} = 0.25 \min(\tau_1, \tau_2, \tau_3)$$

Then uses $N_\text{sub} = \lceil \Delta t / \Delta t_\text{req} \rceil$ sub-steps with explicit **Gauss-Seidel** update (liner first, then middle using new liner value, then outer using new middle value). Includes linearized inter-layer radiation conductance: $G_\text{rad} = 4\sigma T_\text{avg}^3 / R_\text{rad}$.

2. **Stable implicit solver** (`stable_triple_wall.py`): For well-conditioned systems. Uses **implicit Euler** on the 3x3 coupled system with matrix scaling to handle the large conductance ratio between metal and insulation layers. Falls back to Gauss-Seidel if matrix is singular.

**Inter-layer radiation** between concentric cylinders:

Radiation resistance for each gap:
$$R_\text{rad} = \frac{1-\epsilon_i}{\epsilon_i A_i} + \frac{1}{A_i} + \frac{1-\epsilon_o}{\epsilon_o A_o}$$

**Bounds**: $T_1 \in [T_\text{amb}, T_g]$, $T_2 \in [T_\text{amb}, T_1]$, $T_3 \in [T_\text{amb}, T_2]$ (monotonic decrease from gas to ambient).

## 2H. Coupling Strategy (Picard / Fixed-Point)

### Why coupling is needed

The physics creates a circular dependency:

$$T_g \xrightarrow{\text{ideal gas}} \rho \xrightarrow{\text{buoyancy}} \Delta P \xrightarrow{\text{momentum}} \dot{m} \xrightarrow{\text{convection}} h \xrightarrow{\text{energy}} T_g$$

This loop cannot be solved sequentially in one pass because each quantity depends on the others. Picard iteration resolves this by repeatedly cycling through the loop until convergence.

### Residual definitions

Three normalized residuals (`coupled_solver.py:103`):

$$r_T = \frac{\|T^{k+1} - T^k\|_2}{\|T^{k+1}\|_2 + 10^{-10}}$$

$$r_{\dot{m}} = \frac{|\dot{m}^{k+1} - \dot{m}^k|}{|\dot{m}^{k+1}| + 10^{-10}}$$

$$r_P = \frac{\|P^{k+1} - P^k\|_2}{\|P^{k+1}\|_2 + 10^{-10}}$$

Convergence criterion: $\max(r_T, r_{\dot{m}}, r_P) < \text{tol}$ (default $10^{-3}$).

### Relaxation strategy

Standard linear under-relaxation applied to all three state variables:

$$x^{k+1}_\text{relaxed} = \alpha \, x^{k+1}_\text{computed} + (1 - \alpha) \, x^k$$

with adaptive $\alpha$:
- Initial: $\alpha = 0.3$
- If residuals increasing (after iteration 2): $\alpha \leftarrow 0.8\alpha$ (reduce, floor at 0.2)
- If residuals decreasing: $\alpha \leftarrow 1.1\alpha$ (increase, ceiling at 0.5)

### Safety features

- **Runaway detection**: If $\max(T_g) > 1073$ K during iteration, halve $\alpha$ (floor at 0.1).
- **Stall detection**: If iteration > 5 and max residual > 0.1, accept partial convergence and break.
- **First timestep**: Uses sequential (non-coupled) fallback to avoid instability during flow establishment.

### What gets updated each iteration

| Quantity | Updated from | Held fixed during |
|---|---|---|
| Densities $\rho$ | Current $T_g^k$ and part's $P_a$ | -- |
| Buoyancy $\Delta P_\text{buoy}$ | Current $\rho^k$ | -- |
| $\dot{m}$, $P$ profile | Current buoyancy, friction, resistances | ODE starting point (prev timestep $\dot{m}^{n-1}$) |
| Velocities $v$ | Current $\dot{m}^k$, $\rho^k$ | -- |
| $T_g$, wall temps | Current $v^k$, $\dot{m}^k$, wall temps$^{k-1}$ | Segment lengths, material props |

---

# 3. Exact Timestep Algorithm (What Happens, In Order)

This is the **most important section** for the reviewer. It traces the exact order of operations as implemented in the code.

## Pre-loop: Initialization
- Arrays allocated: `_initialize_arrays()` (`chimney.py:1760`)
- Initial conditions set: `_set_initial_conditions()` (`chimney.py:1804`)

## For each timestep `step` from 1 to `time_steps`:

### 0. Update boundary conditions (`chimney.py:821-849`)

0a. **Door opening**: If `door_opening_func` is defined, evaluate it at current time. Call `_update_door_opening(new_opening)` which recalculates $A_\text{eff}$ and $K_\text{equiv}$.  
0b. **Room plenum**: If enabled, call `room_plenum.update(mdot_prev, P_ambient, dt)` to get updated room depressurization.  
0c. **Inlet temperature**: Call `_get_inlet_temp(t, T_ambient)`. Priority: (1) `inlet_temp_func`, (2) CSV interpolation, (3) default exponential ramp. Apply door temperature dilution if enabled.

### 1. Call coupled solver: `solve_coupled_timestep(chimney, step, T_inlet, dt)` (`coupled_solver.py:156`)

**Special case**: If `step < 1`, use sequential fallback (`_solve_sequential`): calls `chimney._solve_comprehensive_heat_transfer(step, T_inlet)` then `chimney._update_flow_and_pressure(step)` and returns.

**Normal case**: Initialize working state from step $n-1$ arrays: copy `T_gas[step-1]`, `mass_flow_rate[step-1]`, `pressure[step-1]`. Set $T_g[0] = T_\text{inlet}$ (inlet BC).

### 2. Picard iteration loop (up to `max_iterations`, default 5):

For each iteration `k`:

> **2.1** `_update_densities()` (`coupled_solver.py:309`)  
> For each segment: $\rho_\text{seg}^k = P_a / (R_\text{air} \cdot T_{g,\text{seg}}^k)$  
> Read: $T_g^k$ (current iteration)  
> Write: $\rho^k$  
>
> **2.2** `_calculate_buoyancy()` (`coupled_solver.py:320`)  
> For each segment: $\Delta P_\text{buoy}[\text{seg}] = g \, \Delta z \, (\rho_\text{amb} - \rho^k_\text{seg})$  
> Read: $\rho^k$, ambient conditions  
> Write: buoyancy array  
>
> **2.3** `_solve_momentum_improved()` (`coupled_solver.py:565`)  
> 2.3.1. Sum buoyancy, subtract wind cap $\Delta P$ and room depressurization $\to$ $\Delta P_\text{drive}$  
> 2.3.2. Accumulate pipe friction resistance (per segment: compute Re, friction factor $f$, K-factors from elbows)  
> 2.3.3. Add inlet restriction resistance ($R_\text{door} + R_\text{stove}$)  
> 2.3.4. Add catalytic bypass resistance (if configured)  
> 2.3.5. (If enabled) Iterate distributed leakage: compute $P_\text{gauge}$ per segment, orifice model for leak flow, relax  
> 2.3.6. (If enabled) Compute acceleration loss, subtract from $\Delta P_\text{drive}$  
> 2.3.7. **Solve momentum balance**: quasi-steady algebraic + lag, or unsteady ODE semi-implicit Euler  
> 2.3.8. Add leakage mass flow to $\dot{m}$. Clip to $\pm 50$ m/s equivalent.  
> 2.3.9. Update pressure profile: march from inlet ($P[0] = 101325$ Pa) segment by segment.  
> Read: buoyancy, $\rho^k$, $\dot{m}^{n-1}$ (for unsteady ODE), previous Picard $\dot{m}^{k-1}$ (for quasi-steady)  
> Write: $\dot{m}^k$, $P^k$ profile  
>
> **2.4** Update velocities  
> For each segment: $v_\text{seg}^k = \dot{m}^k / (\rho_\text{seg}^k \cdot A_\text{seg})$  
> Read: $\dot{m}^k$, $\rho^k$  
> Write: $v^k$  
>
> **2.5** `_solve_energy_improved()` (`coupled_solver.py:912`)  
> For each segment (from 0 to $N_\text{seg}-1$):  
>> 2.5.1. Read previous wall temps from `step-1` arrays (or current iteration if already updated)  
>> 2.5.2. Build `segment_data` dict with gas temp, velocity, wall temps, material props  
>> 2.5.3. Determine $T_\text{upstream}$ from upwind scheme (flow direction dependent)  
>> 2.5.4. Call `comprehensive_ht.calculate_segment_heat_transfer(segment_data, dt)`:  
>>> - Compute gas properties at $T_g$  
>>> - Compute Re, Pr, forced convection Nu  
>>> - **Condensation sub-iteration** (up to 5 iterations):  
>>>   - Compute natural convection (Gr, Ra, Nu) using current $T_w$  
>>>   - Combine forced + natural convection $\to$ $h_i$  
>>>   - Apply engineering minimum $h_i$  
>>>   - Update condensation state (dew point, condensation rate, film modification)  
>>>   - Compute $h_o$ (external natural convection + radiation to ambient)  
>>>   - **Update wall temperatures** (single: semi-implicit; triple: implicit or adaptive sub-stepping)  
>>>   - Check wall temperature convergence; if converged, break  
>>  
>> 2.5.5. Store wall temps into `chimney.T_wall[step, seg]` (and liner/middle/outer for triple)  
>> 2.5.6. Store $h_i$, $h_o$ into `chimney.h_conv_inner[step, seg]`, `chimney.h_conv_outer[step, seg]`  
>> 2.5.7. Compute gas $m_\text{gas}$, $c_p$  
>> 2.5.8. Update gas temperature:  
>>   - If flowing & not inlet: use relaxation model or NTU model  
>>   - If inlet boundary: fixed at $T_\text{inlet}$ (normal) or $T_\text{amb}$ (backdraft)  
>>   - If no/low flow: equilibrate toward wall  
>> 2.5.9. Apply physical bounds: $T_g \in [T_\text{amb}, 1073.15]$ K for non-inlet segments  
>  
> Read: $v^k$, $\dot{m}^k$, wall temps from step-1, $T_g^k$ (inlet), $T_g^{k-1}$ (other segs)  
> Write: wall temps at step, $T_g^k$ (new), $h_i$, $h_o$  
>
> **2.6** Adaptive relaxation adjustment (`coupled_solver.py:253`)  
> After iteration 2: if residuals increasing, $\alpha \leftarrow 0.8\alpha$; if decreasing, $\alpha \leftarrow 1.1\alpha$  
>
> **2.7** Apply relaxation to all state variables (`coupled_solver.py:264`)  
> $T^{k+1} = \alpha T^k_\text{computed} + (1-\alpha) T^{k-1}$  
> $\dot{m}^{k+1} = \alpha \dot{m}^k_\text{computed} + (1-\alpha) \dot{m}^{k-1}$  
> $P^{k+1} = \alpha P^k_\text{computed} + (1-\alpha) P^{k-1}$  
>
> **2.8** Compute convergence residuals (`coupled_solver.py:272`)  
> If $\max(r_T, r_{\dot{m}}, r_P) < \text{tol}$: mark converged, break  
> If runaway detected ($T_\text{max} > 1073$ K): halve $\alpha$  
> If stalled (iter > 5, max residual > 0.1): accept and break  

### 3. Commit converged state (`coupled_solver.py:301`)

Write final Picard state into chimney arrays at time index `step`:
- `T_gas[step]`, `mass_flow_rate[step]`, `pressure[step]`, `density[step]`, `velocity[step]`
- (Wall temperatures were already written directly into step arrays during energy solve)

### 4. Post-step checks (`chimney.py:874-934`)
- Backdraft detection: monitor mass flow sign changes
- Early diagnostics: check for sub-ambient temperatures, NaN, etc. (first 30 steps)
- Progress reporting: every 5% of simulation

---

# 4. Assumptions and Model Limitations

## 1-D approximation
- Single gas temperature per segment. No radial temperature profile in the gas.
- Uniform velocity across each cross-section (plug flow). No boundary layer resolution.
- Lumped wall temperature per layer per segment (no circumferential variation).

## Quasi-steady assumptions
- In the default `quasi_steady` flow model, the momentum equation is algebraic with a first-order lag -- there is no true inertial term. The `unsteady_momentum` mode adds this.
- Gas properties ($c_p$, $\mu$, $k$) are treated as functions of local temperature only. Pressure dependence of density is via ideal gas; other properties ignore pressure.

## Property model limitations
- Gas is treated as **dry air** -- no species transport, no combustion products (CO$_2$, H$_2$O) modifying $c_p$ or $\rho$. The humidity ratio appears only in the condensation model.
- Viscosity via Sutherland's formula is accurate to ~1% for air up to ~1000 K. Beyond that it degrades.
- The $c_p(T)$ polynomial is fit for ~200-800 K. Extrapolation outside this range is uncontrolled.
- Ceramic fiber $k(T)$ uses a simple quadratic fit, not manufacturer-specific data.

## Known numerical stabilizers
- **Re floor**: Reynolds number is floored at 100 (momentum solve) or 10 (pressure profile) to avoid division-by-zero in friction factor.
- **Temperature clipping**: Gas temperature bounded to $[T_\text{amb}, 1073.15 \text{ K}]$ after each iteration.
- **Wall temperature bounds**: $T_w$ clipped to $[T_\text{amb}, T_g]$ for single wall; monotonic constraint for triple wall.
- **$T_\text{steady}$ bounding**: In the relaxation gas energy model, $T_\text{steady}$ is clipped to $\pm 1$ K of the $[T_\text{up}, T_w]$ range to prevent divergence during flow reversal.
- **Minimum resistance**: $R_\text{total} \ge 10^{-6}$ to prevent division by zero.
- **Velocity cap**: $|v| \le 50$ m/s via mass flow clipping.
- **Minimum $h$ values**: $h_i \ge 5$ or 10 W/(m$^2$K), $h_o \ge 1.0$ W/(m$^2$K).
- **Small-$\Delta t$ damping**: Wall temperature damped (0.7 factor) for $\Delta t < 0.01$ s.

## Features that can be toggled off
- `use_coupled_solver = False` $\to$ sequential (non-iterative) solve.
- `MomentumConfig.use_distributed_leakage = False` $\to$ no pipe joint leakage.
- `MomentumConfig.use_accel_loss = False` $\to$ no acceleration pressure loss.
- `MomentumConfig.flow_model = "quasi_steady"` $\to$ no inertial momentum term.
- `room_plenum.enable = False` $\to$ no room depressurization feedback.
- `WindCapConfig.enable = False` $\to$ no wind effects.
- `inlet_restriction.enable = False` $\to$ no door/damper restriction.
- `AdaptiveTripleWallSolver.include_interlayer_radiation = False` $\to$ conduction-only triple wall.

---

# 5. Validation Hooks / Sanity Checks

## Built-in diagnostics

The `DiagnosticsRecorder` (enabled via `Chimney(enable_diagnostics=True)`) records per-timestep:

**Momentum budget** (`coupled_solver.py:861`):
- `dp_buoyancy`: total buoyancy driving pressure [Pa]
- `dp_wind_cap`: wind cap effect [Pa]
- `dp_room`: room depressurization [Pa]
- `dp_friction`: total pipe friction loss [Pa]
- `dp_fittings`: total fitting/elbow loss [Pa]
- `dp_inlet`: inlet restriction loss [Pa]
- `dp_stove`: stove body loss [Pa]
- `dp_bypass`: catalytic bypass loss [Pa]
- `dp_accel`: acceleration loss [Pa]
- `mdot`: mass flow rate [kg/s]
- `velocity_mean`: mean velocity [m/s]
- `rho_mean`: mean density [kg/m$^3$]

**Energy budget** (`coupled_solver.py:1167`):
- `Q_advection`: advective energy flux [W]
- `Q_wall_total`: total wall heat loss [W]
- `Q_latent`: latent heat from condensation [W]
- `T_inlet`, `T_outlet`, `T_gas_mean`, `T_wall_mean` [K]

**Early diagnostics** (`early_diagnostics.py`): Run for first 30 timesteps, checking for:
- Gas or wall temperatures below ambient or absolute zero
- Negative mass flow (backdraft warning)
- Anomalous elbow losses (negative K, K > 100, NaN/Inf)

## Recommended sanity plots for reviewer

1. **$\Delta P_\text{drive}$ vs. $\Delta P_\text{loss}$ over time**: Verifies momentum balance closure. Use `chimney.plot_available_draft()` or diagnostics `dp_buoyancy` vs. sum of losses.

2. **Mass flow rate time series**: Check for unphysical oscillations, sign changes (backdraft). Use `chimney.mass_flow_rate` array.

3. **Max wall temperature / max gas temperature over time**: Verify monotonic heat-up and physically reasonable magnitudes. Use `chimney.T_gas.max(axis=1)` and `chimney.T_wall.max(axis=1)`.

4. **Energy consistency**: $Q_\text{adv}$ vs. $Q_\text{wall} + Q_\text{latent}$ from diagnostics. At steady state these should approximately balance.

5. **Convergence history**: Track `ConvergenceMetrics` per timestep (iterations to converge, final residuals).

6. **Temperature profiles at selected times**: Use `chimney.plot_temperature_profiles([60, 300, 600])` to verify smooth spatial profiles and reasonable gas-wall temperature differences.

7. **Pressure profile**: `chimney.plot_pressure_profile(-1)` for final-state static and differential pressure vs. height.

## Enabling diagnostics

```python
chimney = Chimney(parts=parts, ..., enable_diagnostics=True)
chimney.run_simulation()

# Access momentum budget
momentum_df = chimney.diagnostics.get_momentum_budget_df()

# Access energy budget  
energy_df = chimney.diagnostics.get_energy_budget_df()
```

Performance counters (temporary instrumentation in `_perf_counters.py`) report:
- Picard iterations per timestep (average)
- HT kernel calls per timestep
- Condensation iterations per kernel call
- Triple-wall sub-step events and total sub-steps

## Conservation check

After simulation, `chimney.check_conservation()` (`chimney.py:1395`) computes:
- Total energy input at inlet
- Energy stored in gas (thermal mass change)
- Energy stored in walls (all layers)
- Energy lost to ambient (residual)
- Reports closure as percentage

---

# 6. Appendix

## A. Variable Name Reference

| Variable | Physical meaning | Units | Array name |
|---|---|---|---|
| $T_g$ | Gas temperature | K | `chimney.T_gas[step, seg]` |
| $T_w$ | Wall temperature (single wall) | K | `chimney.T_wall[step, seg]` |
| $T_1$ | Triple-wall liner temp | K | `chimney.T_liner[step, seg]` |
| $T_2$ | Triple-wall insulation temp | K | `chimney.T_middle[step, seg]` |
| $T_3$ | Triple-wall outer shell temp | K | `chimney.T_outer[step, seg]` |
| $\rho$ | Gas density | kg/m$^3$ | `chimney.density[step, seg]` |
| $v$ | Gas velocity | m/s | `chimney.velocity[step, seg]` |
| $P$ | Static pressure | Pa | `chimney.pressure[step, seg]` |
| $\dot{m}$ | Mass flow rate | kg/s | `chimney.mass_flow_rate[step]` |
| $h_i$ | Inner convection coefficient | W/(m$^2$ K) | `chimney.h_conv_inner[step, seg]` |
| $h_o$ | Outer convection coefficient | W/(m$^2$ K) | `chimney.h_conv_outer[step, seg]` |
| $D_i$ | Inner diameter | m | `part.inner_diameter_m` |
| $A$ | Inner cross-section area | m$^2$ | `part.inner_area_m2` |
| $\Delta z$ | Segment length | m | `chimney.segment_lengths[seg]` |
| $P_a$ | Local atmospheric pressure | Pa | `part.Pa` |
| $T_\text{amb}$ | Local ambient temperature | K | `part.T_ambient_K` |
| $\varepsilon$ | Surface roughness | m | `part.roughness_m` |
| $\omega$ | Humidity ratio | kg/kg | `part.humidity_ratio` |
| $C_d$ | Discharge coefficient (inlet) | -- | `inlet_restriction.Cd` |
| $A_\text{eff}$ | Effective inlet area | m$^2$ | `inlet_restriction.effective_area_m2` |
| $K_\text{equiv}$ | Inlet loss coefficient | -- | `inlet_restriction.K_equiv` |

## B. Configuration Options Reference

| Option | What it toggles | File / Field |
|---|---|---|
| `use_coupled_solver` | Picard coupled vs. sequential solve | `chimney.py` constructor |
| `MomentumConfig.flow_model` | `"quasi_steady"` vs. `"unsteady_momentum"` | `coupled_solver.py:30` |
| `MomentumConfig.gas_energy_model` | `"current"` (relaxation) vs. `"plugflow_ntu"` | `coupled_solver.py:31` |
| `MomentumConfig.friction_method` | `"current"` (Blasius/Swamee-Jain) vs. `"colebrook"` | `coupled_solver.py:27` |
| `MomentumConfig.distributed_draft_model` | Master switch enabling leakage + accel loss | `coupled_solver.py:20` |
| `MomentumConfig.use_distributed_leakage` | VENT-II distributed leakage | `coupled_solver.py:21` |
| `MomentumConfig.use_accel_loss` | Acceleration pressure loss term | `coupled_solver.py:22` |
| `inlet_restriction.enable` | Door/damper restriction at inlet | `chimney.py:24` |
| `inlet_restriction.inlet_area_mode` | `"linear"` vs. `"quadrature"` area combination | `chimney.py:97` |
| `catalytic_bypass.bypass_state` | `"open"`, `"closed"`, or float 0-100 | `chimney.py:130` |
| `WindCapConfig.enable` | Wind effects at chimney cap | `wind_cap_effects.py:18` |
| `WindCapConfig.cap_type` | Cap geometry (`"venturi"`, `"rain_cap"`, etc.) | `wind_cap_effects.py:23` |
| `RoomPlenumConfig.enable` | Room depressurization feedback | `room_plenum.py:45` |
| `LeakageConfig.enable` | Pipe joint leakage | `chimney.py:158` |
| `enable_diagnostics` | Record momentum/energy budgets | `chimney.py` constructor |
| `part.type` | `"single"` or `"triple"` wall | `chimney.py:188` |
| `part.corrugation` | Spiral or annular corrugation dict | `chimney.py:250` |

## C. Code Map

| Physics function | Module | Key lines |
|---|---|---|
| Ideal gas density | `physics.py` | `air_density()` :23 |
| Sutherland viscosity | `physics.py` | `air_viscosity()` :42 |
| Thermal conductivity | `physics.py` | `air_thermal_conductivity()` :65 |
| Specific heat | `physics.py` | `air_specific_heat()` :84 |
| Ceramic fiber k(T) | `physics.py` | `ceramic_fiber_thermal_conductivity()` :104 |
| Friction factor | `mass_flow.py` | `friction_factor()` :18 |
| Colebrook-White | `mass_flow.py` | `_friction_factor_colebrook()` :63 |
| Distributed leakage | `mass_flow.py` | `calculate_leakage_mass_flow()` :101 |
| Acceleration loss | `mass_flow.py` | `calculate_accel_pressure_loss()` :148 |
| Picard coupling loop | `coupled_solver.py` | `solve_coupled_timestep()` :156 |
| Density update | `coupled_solver.py` | `_update_densities()` :309 |
| Buoyancy calculation | `coupled_solver.py` | `_calculate_buoyancy()` :320 |
| Inlet restriction R | `coupled_solver.py` | `_calculate_inlet_restriction_resistance()` :341 |
| Momentum solve | `coupled_solver.py` | `_solve_momentum_improved()` :565 |
| Energy solve | `coupled_solver.py` | `_solve_energy_improved()` :912 |
| Forced convection Nu | `comprehensive_heat_transfer.py` | `nusselt_forced_convection_internal()` |
| Natural convection Nu | `comprehensive_heat_transfer.py` | `nusselt_natural_convection_internal()` |
| Combined h | `comprehensive_heat_transfer.py` | `combined_convection_coefficient()` |
| Segment HT kernel | `comprehensive_heat_transfer.py` | `calculate_segment_heat_transfer()` :711 |
| Single-wall transient | `comprehensive_heat_transfer.py` | `transient_wall_temperature()` :524 |
| Triple-wall transient | `comprehensive_heat_transfer.py` | `transient_triple_wall_temperatures()` :602 |
| Triple-wall stable solver | `stable_triple_wall.py` | `solve_triple_wall_stable()` |
| Triple-wall adaptive | `adaptive_triple_wall.py` | `solve_with_adaptive_dt()` :142 |
| Wind cap pressure | `wind_cap_effects.py` | `wind_cap_deltaP()` :112 |
| Room plenum dynamics | `room_plenum.py` | `RoomPlenum.update()` |
| Condensation physics | `condensation_physics.py` | `CondensationPhysics` class |
| Spiral corrugation | `corrugation_effects.py` | `spiral_equivalents()` |
| Draft analysis | `draft_analysis.py` | `DraftAnalyzer.calculate_draft()` |
| Early diagnostics | `early_diagnostics.py` | `EarlyTimeDiagnostics.check_timestep()` |
| Chimney orchestrator | `chimney.py` | `Chimney.run_simulation()` :795 |
| Conservation check | `chimney.py` | `check_conservation()` :1395 |